# Setup

In [2]:
from pprint import pprint
import os
import numpy as np
import torch
from transformers.utils import logging
from transformers import set_seed

# Seed
seed = 42
set_seed(seed)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_bf16 = (device.type == 'cuda') and (torch.cuda.is_bf16_supported())

# Suppress hf loggings
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
logging.set_verbosity_error()
logging.disable_progress_bar()

W1019 18:36:55.124000 24012 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1019 18:36:55.196000 24012 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1019 18:36:56.135000 24012 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


# 1. RAG

In [2]:
import wikipediaapi
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from qdrant_client import models
from sentence_transformers import CrossEncoder
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain.agents import create_agent


# 1. Load: one Wikipedia page.
wiki = wikipediaapi.Wikipedia(
    user_agent='LangChainRAG/1.0',
    language='en',
)
page = wiki.page('Retrieval-augmented generation')
documents = [
    Document(page_content=page.text, metadata={'source': page.fullurl})
]

# 2. Split
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
)
chunks = splitter.split_documents(documents)

# 3. Embedding
dense_emb = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
)
sparse_emb = FastEmbedSparse(model_name='Qdrant/bm25')

# 4. Vector DB: store both dense and sparse vectors.
vector_db = QdrantVectorStore.from_documents(
    collection_name='wiki_hybrid',
    location=':memory:',    # in-memory.
    documents=chunks,
    embedding=dense_emb,
    sparse_embedding=sparse_emb,
    sparse_vector_params={'modifier': models.Modifier.IDF},
    retrieval_mode=RetrievalMode.HYBRID,    # hybrid retrieval: dense + sparse.
)

# 5. Retriever and reranker.
retriever = vector_db.as_retriever(search_kwargs={"k": 10})
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

# 6. Retrieval Tool: retrieve 10 candidates, rerank, and return the best 3.
@tool
def retrieve_context(query: str) -> str:
    """Search the Wikipedia article about retrieval-augmented generation."""
    retrieved_docs = retriever.invoke(query)
    if not retrieved_docs:
        return "No documents found."

    ranked = reranker.rank(
        query,
        [doc.page_content for doc in retrieved_docs],
        top_k=3,
    )
    reranked_docs = [retrieved_docs[item["corpus_id"]] for item in ranked]

    return "\n\n".join(
        f"Source: {doc.metadata['source']}\n{doc.page_content}"
        for doc in reranked_docs
    )


# 7. Template
system_prompt = """You are a helpful assistant.
Answer precisely and shortly."""
template = ChatPromptTemplate.from_messages(
    [
        ('system', system_prompt),
        ('human', "Query: {query}"),
    ]
)

# 8. Prompt
query = 'What is RAG?'
prompt = template.invoke({'query': query})

# 9. Chat model
chat = ChatOllama(
    model='qwen3:4b',
    reasoning=True,
    seed=seed,
)

# 10. Agent
agent = create_agent(
    model=chat,
    tools=[retrieve_context],
)

# 11. Response
response = agent.invoke({'messages': prompt.to_messages()})

for message in response['messages']:
    print(f"[{type(message).__name__}]\n{message.content}", end='\n\n')

[SystemMessage]
You are a helpful assistant.
Answer precisely and shortly.

[HumanMessage]
Query: What is RAG?

[AIMessage]


[ToolMessage]
Source: https://en.wikipedia.org/wiki/Retrieval-augmented_generation
Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. With RAG, LLMs first refer to a specified set of documents, then respond to user queries. These documents supplement information from the LLM's pre-existing training data. This allows LLMs to use domain-specific and/or updated information that is not available in the training data. For example, this enables LLM-based chatbots to access internal company data or generate responses based on authoritative sources. The technique was first proposed in 2020 and has since become a widely adopted approach in modern AI systems.

Source: https://en.wikipedia.org/wiki/Retrieval-augmented_generation
RAG improves LLMs by incorporat

# 2. Message State

## 2.1. Basic

In [3]:
from langchain_ollama import ChatOllama
from langgraph.graph import MessagesState, StateGraph, START, END
from langchain_core.messages import SystemMessage, HumanMessage


# Chat model
chat = ChatOllama(
    model='qwen3:4b',
    reasoning=True,
    seed=seed,
)

# Node: state -> changed state.
def call_model(state: MessagesState):
    response = chat.invoke(state['messages'])   # automatically concatenates.

    return {'messages': [response]}


# Graph
builder = StateGraph(MessagesState)
builder.add_node('chat', call_model)
builder.add_edge(START, 'chat')
builder.add_edge('chat', END)

graph = builder.compile()

# Generate
result = graph.invoke(
    input={
        'messages': [
            SystemMessage("You are a helpful assistant. Answer shortly."),
            HumanMessage("My name is yana."),
            HumanMessage("I was born in 2026."),
            HumanMessage("What is my name?"),
            HumanMessage("In which month was I born?")
        ]
    },
)

for message in result['messages']:
    print(f"{type(message).__name__}")
    print(message.content, end='\n\n')

SystemMessage
You are a helpful assistant. Answer shortly.

HumanMessage
My name is yana.

HumanMessage
I was born in 2026.

HumanMessage
What is my name?

HumanMessage
In which month was I born?

AIMessage
Your name is **Yana**.  

You were born in **2026** (a future year, as of 2024), so this date is impossible.  
**Month of birth**: Not specified in your message.  

*(Answering briefly as requested)*



## 2.2. In-Memory

- Within one process: removes after restart.
- Within one chat thread.

In [4]:
from langchain_ollama import ChatOllama
from langgraph.graph import MessagesState, StateGraph, START, END
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


# Chat model
chat = ChatOllama(
    model='qwen3:4b',
    reasoning=True,
    seed=seed,
)

# Node: state -> changed state.
def call_model(state: MessagesState):
    response = chat.invoke(state['messages'])       # automatically concatenates the previous conversation.

    return {'messages': [response]}

# Graph with checkpoint
builder = StateGraph(MessagesState)
builder.add_node('chat', call_model)
builder.add_edge(START, 'chat')
builder.add_edge('chat', END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

# Config
config = {
    'configurable': {
        'thread_id': 'conversation-1'       # thread id -> memory only perisists within this thread.
    }
}

# Conversation 1
system_prompt = """You are a helpful assistant.
Answer precisely and shortly."""

result = graph.invoke(
    input={
        'messages': [
            SystemMessage(system_prompt),
            HumanMessage("My name is yana. Remember this.")
        ],
    },
    config=config,
)
print(f"First: {result['messages'][-1].content}", end='\n\n')


# Conversation 2
result = graph.invoke(
    input={
        'messages': [
            SystemMessage(system_prompt),
            HumanMessage("What is my name?")
        ],
    },
    config=config,  # same thread as conversation 1.
)
print(f"Second: {result['messages'][-1].content}", end='\n\n')


# Memory state
snapshot = graph.get_state(config)

print(f"Config: {snapshot.config}")
print(f"Created at: {snapshot.created_at}", end='\n\n')

for message in snapshot.values['messages']:
    print(f"[{type(message).__name__}]\n{message.content}", end='\n\n')

First: Remembered: Yana

Second: Yana

Config: {'configurable': {'thread_id': 'conversation-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1cba06-5fad-6df3-8004-4722571aa876'}}
Created at: 2026-10-19T09:35:19.134144+00:00

[SystemMessage]
You are a helpful assistant.
Answer precisely and shortly.

[HumanMessage]
My name is yana. Remember this.

[AIMessage]
Remembered: Yana

[SystemMessage]
You are a helpful assistant.
Answer precisely and shortly.

[HumanMessage]
What is my name?

[AIMessage]
Yana



## 2.3. In-Thread

- Across many process: persists after restart.
- Within one chat thread.

In [5]:
from langchain_ollama import ChatOllama
from langgraph.graph import MessagesState, StateGraph, START, END
from langchain_core.messages import SystemMessage, HumanMessage
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver


# Chat model
chat = ChatOllama(
    model='qwen3:4b',
    reasoning=True,
    seed=seed,
)

# Node: state -> changed state.
def call_model(state: MessagesState):
    response = chat.invoke(state['messages'])       # automatically concatenates the previous conversation.

    return {'messages': [response]}

# Graph builder
builder = StateGraph(MessagesState)
builder.add_node('chat', call_model)
builder.add_edge(START, 'chat')
builder.add_edge('chat', END)

# Checkpointer
connection = sqlite3.connect(
    database='../tmp/memory/memory.sqlite',
    check_same_thread=False,    # python execution thread, not chat thread.
)

checkpointer = SqliteSaver(connection)

# Graph
graph = builder.compile(checkpointer=checkpointer)

config = {
    'configurable': {
        'thread_id': 'conversation-1'
    }
}

# Conversation
system_prompt = """You are a helpful assistant.
Answer precisely and shortly."""

result = graph.invoke(
    input={
        'messages': [
            SystemMessage(system_prompt),
            # HumanMessage("My name is yana. Remember this."),        # 1st run.
            HumanMessage("What is my name?")                        # 2nd run.
        ],
    },
    config=config,
)
print(f"Result: {result['messages'][-1].content}", end='\n\n')


Result: Yana



## 2.4. In-User

- Across many process: persists after restart.
- Across chat thread: persists within many chat threads for the same user.

In [ ]:
from langchain_ollama import ChatOllama
from langgraph.graph import MessagesState, StateGraph, START, END
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.runnables import RunnableConfig
from langgraph.runtime import Runtime
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.store.sqlite import SqliteStore
from uuid import uuid4
import sqlite3


# Chat model
chat = ChatOllama(
    model='qwen3:4b',
    reasoning=True,
    seed=seed,
)

# Node
def call_model(state: MessagesState, config: RunnableConfig, runtime: Runtime):
    user_id = config['configurable']['user_id']
    namespace = ('users', user_id)

    # Retrieve user memories across threads.
    memories = runtime.store.search(namespace, limit=20)
    memory_text = '\n'.join(m.value['text'] for m in memories)

    response = chat.invoke(
        [SystemMessage(f"Previously saved user information:\n{memory_text}")]
        + state['messages']
    )

    # Save the latest user message as memory.
    last_message = state['messages'][-1]
    runtime.store.put(
        namespace,
        str(uuid4()),   # unique user id across the store.
        {'text': last_message.content},
    )

    return {'messages': [response]}


# Graph builder
builder = StateGraph(MessagesState)
builder.add_node('chat', call_model)
builder.add_edge(START, 'chat')
builder.add_edge('chat', END)

# Checkpointer: in-thread persistence.
connection = sqlite3.connect(
    database='../tmp/memory/memory.sqlite',
    check_same_thread=False,
)
checkpointer = SqliteSaver(connection)

# Store: cross-thread persistence.
store_connection = sqlite3.connect(
    database='../tmp/memory/user_memory.sqlite',
    check_same_thread=False,
    isolation_level=None,   # enable autocommit mode.
)
store = SqliteStore(store_connection)
store.setup()

# Graph
graph = builder.compile(checkpointer=checkpointer, store=store)

config = {
    'configurable': {
        'thread_id': 'conversation-2',      # 'conversation-1' -> 'conversation-2'.
        'user_id': 'user-1',                # unchanged.
    }
}

# Conversation
system_prompt = """You are a helpful assistant.
Answer precisely and shortly."""

result = graph.invoke(
    input={
        'messages': [
            SystemMessage(system_prompt),
            # HumanMessage("My name is yana. Remember this."),  # 1st run.
            HumanMessage("What is my name?"),               # 2nd run.
        ],
    },
    config=config,
)
print(f"Result: {result['messages'][-1].content}", end='\n\n')

Result: Yana



# 3. Memory Representation

# 4. Memory Management

# 5. Memory Retrieval

# 6. Consolidation

# 7. Evaluation

# 8. MemGPT: Multiple Memory Tiers

# 9. Titans: Inference-Learning Memory